# Phase 4 — DSPy Optimization + QLoRA Fine-Tuning (Tier 2, N=10)
Tier 2 baseline (no optimization) was 41.7% — this is the tier with real headroom for DSPy/QLoRA to show a difference, unlike Tier 1 which saturated at 100% for both methods.

**Key difference from Tier 1:** each task needs MULTIPLE tool calls, so the DSPy program runs a bounded loop internally (see `dspy_optimize_tier2.py`), and QLoRA trains on full multi-turn trajectories, not single calls.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!nvidia-smi

**STOP: must show a Tesla T4 GPU table with 0MiB used.**

## 1. Clone repo (safe to re-run any time)

In [ ]:
import shutil, os
os.chdir("/content")
if os.path.exists("agentic-prompt-vs-finetune"):
    shutil.rmtree("agentic-prompt-vs-finetune")
!git clone https://github.com/nive62tech/agentic-prompt-vs-finetune.git
%cd agentic-prompt-vs-finetune
!pwd

In [ ]:
!grep -c "def sample_tier2_training" envs/training_data.py
!grep -c "class ChainProgram" dspy_optimize_tier2.py
!grep -c "def build_chat_example" qlora_finetune_tier2.py

Each of the 3 grep counts above should print `1`. If any print `0`, the corresponding file wasn't pushed correctly — check `git log` on your laptop before continuing.

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets dspy-ai optuna

In [ ]:
from huggingface_hub import login
login()

## 2. Sanity-check the Tier 2 training data generator (no GPU needed)

In [ ]:
!python envs/training_data.py

Check the TIER 2 section near the bottom: pool size ~93, leakage check = 0.

## 3. Load model + run DSPy optimization

In [ ]:
import sys, json
sys.path.insert(0, ".")
from envs.agent_harness import load_model
from envs.dspy_lm import LocalLlamaLM
from envs.training_data import sample_tier2_training
from tasks.tier2 import TIER2_HELDOUT
import dspy_optimize_tier2

model, tok = load_model("meta-llama/Llama-3.1-8B-Instruct")
lm = LocalLlamaLM(model, tok)
print("Model + DSPy LM wrapper ready.")

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

train_10 = sample_tier2_training(10)
print(f"Training on {len(train_10)} Tier 2 examples.")

optimized_program = dspy_optimize_tier2.optimize(lm, train_10)
print("DSPy optimization complete.")

This will likely take LONGER than Tier 1's did — each task now needs multiple generation turns instead of one, so every trial in the optimization loop costs more. If it OOMs despite the baked-in fixes, paste the full error rather than improvising new settings live.

In [ ]:
dspy_results = dspy_optimize_tier2.evaluate_program(optimized_program, TIER2_HELDOUT)
for r in dspy_results:
    print(f"[{'PASS' if r['grade']['success'] else 'FAIL'}] {r['id']} — {r['grade'].get('failure_type')}")

dspy_success_rate = sum(r["grade"]["success"] for r in dspy_results) / len(dspy_results)
print(f"\nDSPy-optimized Tier 2 (N=10) held-out success rate: {dspy_success_rate:.1%}")

with open("results/tier2_dspy_n10_results.json", "w") as f:
    json.dump(dspy_results, f, indent=2)

## 4. Download DSPy results NOW, before continuing

In [ ]:
from google.colab import files
files.download("results/tier2_dspy_n10_results.json")

## 5. Restart the runtime before QLoRA
Do NOT try to free memory in-place and continue in this same session — that caused OOM crashes twice already in earlier phases. Runtime -> Restart session (or Disconnect and delete runtime if that's not enough), then re-run: env var cell, GPU check, Section 1 (clone/verify/install/login). Skip straight to Section 6 after that.

## 6. QLoRA fine-tuning (N=10)

In [ ]:
!python qlora_finetune_tier2.py --n 10

Training examples are longer here (full multi-turn trajectories, not single calls) so this will take longer than Tier 1's N=10 run did. Watch the loss trend downward same as before.

## 7. Evaluate the fine-tuned adapter
This uses the SAME multi-turn agent loop as the Phase 2 baseline (envs/agent_harness.py's run_agent), not the DSPy ChainProgram — the adapter is evaluated the way it'll actually be used, via the standard harness.

In [ ]:
import torch, json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from envs.agent_harness import run_agent
from envs.tools import TOOL_SCHEMAS, call_tool
from tasks.tier2 import TIER2_HELDOUT
from grader import grade_task

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
base_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", quantization_config=bnb_config, device_map={"": 0})
ft_model = PeftModel.from_pretrained(base_model, "adapters/tier2_n10")
ft_tok = AutoTokenizer.from_pretrained("adapters/tier2_n10")
print("Fine-tuned model loaded.")

In [ ]:
qlora_results = []
for task in TIER2_HELDOUT:
    tool_calls, final_text = run_agent(ft_model, ft_tok, task["prompt"], TOOL_SCHEMAS, call_tool, max_turns=6)
    grade = grade_task(task, tier=2, tool_calls=tool_calls, final_text=final_text)
    qlora_results.append({
        "id": task["id"], "prompt": task["prompt"],
        "tool_calls": tool_calls, "final_text": final_text, "grade": grade,
    })
    print(f"[{'PASS' if grade['success'] else 'FAIL'}] {task['id']} — {grade.get('failure_type')}")

qlora_success_rate = sum(r["grade"]["success"] for r in qlora_results) / len(qlora_results)
print(f"\nQLoRA fine-tuned Tier 2 (N=10) held-out success rate: {qlora_success_rate:.1%}")

with open("results/tier2_qlora_n10_results.json", "w") as f:
    json.dump(qlora_results, f, indent=2)

## 8. Download everything and push

In [ ]:
from google.colab import files
files.download("results/tier2_qlora_n10_results.json")
files.download("adapters/tier2_n10/training_examples.json")

Move both into `results/` on your laptop (rename `training_examples.json` to `tier2_training_examples_n10.json`), then:
```bash
git add results/tier2_dspy_n10_results.json results/tier2_qlora_n10_results.json results/tier2_training_examples_n10.json
git commit -m "Phase 4: DSPy + QLoRA results for Tier 2, N=10"
git push
```